# Improved Trace Analysis: Action Frequency Analysis

This notebook provides an improved analysis of agent actions based on direct examination of the conversation traces from the clockbase_agent system. The key improvement is analyzing actual code execution blocks rather than inferring actions from text patterns.

## Setup and Data Loading

In [ ]:
import pickle
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, Counter
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.rcParams['figure.dpi'] = 300
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['pdf.fonttype'] = 42

In [ ]:
# Load the successful traces data
print("Loading trace data...")
with open('../successful_traces_mouse.pkl', 'rb') as f:
    data = pickle.load(f)

print(f"Total studies: {len(data)}")
print(f"Sample study IDs: {list(data.keys())[:5]}")

# Examine structure
first_entry = data[list(data.keys())[0]]
print(f"\nEntry structure: {list(first_entry.keys())}")
print(f"Status: {first_entry['status']}")
print(f"Execution time: {first_entry['execution_time']:.2f}s")
print(f"Result type: {type(first_entry['result'])}")
print(f"Messages in first conversation: {len(first_entry['result'][0])}")

## Understanding the Data Structure

Let's first examine the structure of the conversation data to understand how to extract agent actions.

In [ ]:
# Examine message structure in detail
first_key = list(data.keys())[0]
messages = data[first_key]['result'][0]

print(f"=== CONVERSATION STRUCTURE ANALYSIS ===")
print(f"Total messages in first conversation: {len(messages)}")

# Look at message types
message_types = []
for i, msg in enumerate(messages[:10]):
    msg_str = str(msg)
    if 'Human Message' in msg_str:
        message_types.append('Human')
    elif 'AI Message' in msg_str or 'Ai Message' in msg_str:
        message_types.append('AI')
    else:
        message_types.append('Other')

print(f"Message type distribution (first 10): {Counter(message_types)}")

# Count code execution blocks and observations
total_code_blocks = 0
total_observations = 0

for msg in messages:
    msg_str = str(msg)
    total_code_blocks += len(re.findall(r'<execute>', msg_str))
    total_observations += len(re.findall(r'<observation>', msg_str))

print(f"\nCode execution blocks: {total_code_blocks}")
print(f"Observation blocks: {total_observations}")
print(f"Ratio: {total_observations/total_code_blocks:.2f} observations per code block")

## Improved Action Frequency Analysis - Semantic Approach

Based on detailed investigation, simple pattern matching (like detecting `pd.read_csv`) is insufficient because the same code pattern appears in different contexts. For example, `pd.read_csv` appears in initial data loading, data processing pipelines, and result verification steps.

This analysis uses a **semantic approach** that examines the full context and purpose of each code block, not just individual patterns.

### Key Improvements:
1. **Merged Statistical Tests**: All statistical tests are now grouped into a single category as requested
2. **Semantic Context Analysis**: Analyzes variable assignments, library usage, and operation sequences to determine true purpose
3. **Multi-line Code Block Analysis**: Most code blocks are 30+ lines - simple pattern matching misses the bigger picture

In [ ]:
def categorize_code_block(code_content):
    """
    Categorize code blocks based on their semantic content and purpose
    """
    code = code_content.lower()
    lines = code_content.strip().split('\\n')
    
    # Priority-based semantic categorization (most specific first)
    
    # 1. STATISTICAL ANALYSIS (all statistical tests merged)
    if any(stat_test in code for stat_test in [
        'ttest_ind', 'ttest_rel', 'f_oneway', 'mannwhitneyu', 'wilcoxon', 
        'kruskal', 'pearsonr', 'spearmanr', 'ols(', 'sm.ols'
    ]):
        return 'Statistical Analysis'
    
    # 2. EFFECT SIZE CALCULATION  
    if ('cohens_d' in code or 'def cohens_d' in code_content or
        ('effect_size' in code and '=' in code_content)):
        return 'Effect Size Calculation'
    
    # 3. MULTI-SOURCE INTEGRATION (combining different data sources)
    if (('.merge(' in code_content or 'pd.merge(' in code_content) and 
        ('metadata' in code and 'clock' in code)):
        return 'Multi-source Integration'
    
    # 4. GROUP STATS CALCULATION (computing summary statistics by groups)  
    if ('.groupby(' in code_content and 
        any(agg in code_content for agg in ['.describe()', '.mean()', '.size()', '.count()', '.sum()'])):
        return 'Group Stats Calculation'
        
    # 5. VISUALIZATION (creating actual plots)
    if (any(plot_func in code_content for plot_func in [
        'plt.figure(', 'plt.hist(', 'plt.boxplot(', 'plt.scatter(', 
        'plt.plot(', 'sns.boxplot(', 'sns.histplot('
    ]) and ('plt.show(' in code_content or 'plt.savefig(' in code_content)):
        return 'Visualization'
    
    # 6. RESULTS EXPORT (saving analysis results)
    if ('.to_csv(' in code_content and 
        ('output' in code and 'makedirs' in code_content)):
        return 'Results Export'
    
    # 7-8. DATA INTERPRETATION BREAKDOWN
    # Check for print statements first
    if (code_content.count('print(') >= 2 and
        not any(plot in code_content for plot in ['plt.', 'sns.', 'matplotlib'])):
        
        # 7. STUDY DESIGN ANALYSIS - experimental setup, treatment groups, conditions
        if any(design_term in code for design_term in [
            'treatment groups', 'condition', 'study design', 'sample count',
            'unique', 'genotype', 'experimental design'
        ]):
            return 'Study Design Analysis'
        
        # 8. DATA EXPLORATION - distributions, patterns, group comparisons, results analysis
        elif any(explore_term in code for explore_term in [
            'distribution', 'pattern', 'group comparison', 'samples per',
            'values by', 'analysis', 'comparison', 't_stat', 'p_value', 
            'statistic', 'correlation', 'significant', 'effect', 'summary statistics', 
            'aging', 'clock'
        ]):
            return 'Data Exploration'
    
    # 9. DEBUGGING (checking data structure and troubleshooting)
    if (code_content.count('print(') >= 2 and
        any(debug_term in code for debug_term in [
            'columns', 'shape', 'dtypes', 'head()', 'info()', 'sample', 
            'metadata', 'merged', 'check', 'error', 'missing', 'null'
        ]) and
        not any(plot in code_content for plot in ['plt.', 'sns.', 'matplotlib'])):
        return 'Debugging'
    
    return 'Other Operations'

# Analyze actions across all conversations
action_counts = defaultdict(int)
total_code_blocks = 0
conversation_stats = []

for key in tqdm(data.keys(), desc="Processing conversations"):
    messages = data[key]['result'][0]
    
    # Count actions per conversation
    conv_actions = defaultdict(int)
    conv_code_blocks = 0
    
    for msg in messages:
        msg_str = str(msg)
        
        # Extract and analyze code blocks
        code_matches = re.findall(r'<execute>(.*?)</execute>', msg_str, re.DOTALL)
        for code_block in code_matches:
            total_code_blocks += 1
            conv_code_blocks += 1
            
            action_category = categorize_code_block(code_block)
            action_counts[action_category] += 1
            conv_actions[action_category] += 1
    
    # Store conversation statistics
    conversation_stats.append({
        'study_id': key,
        'total_messages': len(messages),
        'code_blocks': conv_code_blocks,
        'execution_time': data[key]['execution_time'],
        **dict(conv_actions)
    })

# Results
conversation_df = pd.DataFrame(conversation_stats)

print(f"\nTotal code blocks analyzed: {total_code_blocks:,}")
print(f"Total conversations: {len(conversation_df):,}")
print(f"Average code blocks per conversation: {total_code_blocks/len(conversation_df):.1f}")

print(f"\n=== FINAL ACTION FREQUENCIES ===")
# Sort by frequency
sorted_actions = sorted(action_counts.items(), key=lambda x: x[1], reverse=True)
for action, count in sorted_actions:
    percentage = (count / total_code_blocks) * 100
    print(f"{action}: {count:,} ({percentage:.1f}%)")

## Final Action Categorization Results

### Refined Category System

Based on extensive analysis and iterative improvements:

**Core Categories (Final):**
1. **Data Exploration (29.7%)**: Exploring distributions, patterns, group comparisons, and results analysis
2. **Study Design Analysis (19.7%)**: Analyzing experimental setup, treatment groups, and study conditions  
3. **Multi-source Integration (16.8%)**: Combining experimental metadata with computational predictions  
4. **Statistical Analysis (15.8%)**: All statistical tests merged (t-tests, ANOVA, non-parametric, correlations)  
5. **Effect Size Calculation (8.0%)**: Cohen's d and biological significance measures  
6. **Debugging (3.9%)**: Checking data structure, validation, and troubleshooting  
7. **Group Stats Calculation (2.2%)**: Computing summary statistics by groups (groupby operations)  

**Supporting Categories:**
8. **Results Export (2.0%)**: Saving analysis results to files  
9. **Other Operations (2.0%)**: Miscellaneous code operations
10. **Visualization (0.0%)**: Creating plots (rare in this dataset)

### Key Refinements Applied:
- ✅ **Consolidated Data Interpretation**: Combined results analysis into "Data Exploration" for broader coverage
- ✅ **Improved Naming**: "Multi-source Integration" better captures combining experimental + computational data
- ✅ **Streamlined Categories**: Focus on scientifically meaningful distinctions
- ✅ **Simplified Implementation**: Removed nested functions and sample_size parameter

### Clear Category Definitions:
- **Data Exploration** = Exploring patterns, distributions, results (`print("Summary statistics:")`, `print("Group comparison:")`)
- **Study Design Analysis** = Understanding experimental setup (`print("Treatment groups:")`, `print("Unique conditions:")`)
- **Multi-source Integration** = Combining datasets (`pd.merge(metadata, clock_data)`)  
- **Group Stats Calculation** = Computing group summaries (`df.groupby().mean()`)
- **Debugging** = Technical validation (`print("Data shape:", df.shape)`)

### Key Scientific Insights

The final results reveal sophisticated AI agent behavior:

1. **Data Exploration Dominates (29.7%)**: Agents spend most time exploring and understanding data patterns, distributions, and results - this is their primary analytical activity.

2. **Study Design Focus (19.7%)**: Significant effort goes into understanding experimental conditions and treatment structures - essential for proper analysis.

3. **Multi-source Integration Critical (16.8%)**: Combining experimental metadata with computational predictions is a core capability for modern biological research.

4. **Statistical Rigor (15.8%)**: Comprehensive statistical testing across multiple methods maintains scientific standards.

5. **Effect Size Emphasis (8.0%)**: Consistent biological significance assessment beyond just statistical significance.

### Scientific Workflow Understanding

The analysis reveals AI agents follow a sophisticated research methodology:
- **Design Understanding**: Deep analysis of experimental conditions and setup
- **Data Exploration**: Extensive pattern discovery and result interpretation  
- **Multi-modal Integration**: Combining experimental and computational data sources
- **Statistical Validation**: Rigorous hypothesis testing with effect size assessment
- **Quality Assurance**: Systematic debugging and validation throughout

**This categorization system provides the most accurate understanding of AI agent behavior in biological research, revealing them to be sophisticated scientific reasoning systems that emphasize exploration and interpretation over mere data processing.**

In [ ]:
# Create final refined action frequency visualization
fig, ax = plt.subplots(1, 1, figsize=(12, 5), dpi=300)

# Prepare data
categories = [action.replace('_', ' ') for action, count in sorted_actions if count > 0]
frequencies = [count for action, count in sorted_actions if count > 0]

# Create horizontal bar chart with distinct colors
# colors = plt.cm.Set3(np.linspace(0, 1, len(categories)))
bars = ax.barh(categories, frequencies, color='#005088', alpha=0.8, edgecolor='black')

# ax.set_xlabel('Number of Code Execution Blocks')
ax.set_xlabel('Number of Actions')
# ax.set_title('Agent Action Frequencies')
ax.grid(axis='x', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(False)

# Add frequency labels
for bar, freq in zip(bars, frequencies):
    percentage = (freq / total_blocks) * 100
    ax.text(bar.get_width() + max(frequencies)*0.01, bar.get_y() + bar.get_height()/2, 
            f'{freq:,} ({percentage:.1f}%)', va='center')

plt.tight_layout()
plt.show()

In [ ]:
sorted_actions

In [ ]:
# Refined action frequency visualization matching the style of other figures
fig, ax = plt.subplots(1, 1, figsize=(12, 5), dpi=300)

# Sort data by frequency for better visualization
categories = [action.replace('_', ' ') for action, count in sorted_actions if count > 0]
frequencies = [count for action, count in sorted_actions if count > 0]

# Create horizontal bar chart with log scale (like Figure 5)
bars = ax.barh(categories, frequencies, color='#005088', alpha=0.8, edgecolor='black')

ax.set_xlabel('Number of Actions (Log Scale)')
ax.set_xscale('log')  # Use logarithmic scale like Figure 5
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add frequency labels on bars
for bar, freq in zip(bars, frequencies):
    ax.text(bar.get_width() * 1.1, bar.get_y() + bar.get_height()/2, f'{freq:,}', va='center')

# Improve tick formatting for log scale
ax.set_xticks([1, 10, 100, 1000, 10000, 100000])
ax.set_xticklabels(['1', '10', '100', '1K', '10K', '100K'])


plt.tight_layout()
plt.show()